# Sheild — Track 3 model training
Train a CPU-only multilingual CRF and calibrate a conservative Fellegi–Sunter-style pair scorer. Upload **Sheild_Training_Package.zip** when prompted. It contains independently generated synthetic data and training source. No organizer practice answers or real patient data are used. Run during the event if required by the organizer original-work rules. The training is separate from the offline evaluation pipeline. These models propose candidates; the hybrid engine and verifier must still be implemented.

In [ ]:
# 1. CPU runtime; install training/inference dependency
import sys, subprocess, os
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-crfsuite==0.9.12'])

In [ ]:
# 2. Upload the training package, then extract only its expected training files.
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io, shutil
uploaded = files.upload()
name = 'Sheild_Training_Package.zip'
assert name in uploaded, f'Upload {name}'
work = Path('/content/sheild_model_training')
work.mkdir(exist_ok=True)
expected = {'generate_dataset.py', 'model_features.py', 'model_runtime.py', 'train_models.py', 'requirements.txt', 'data/manifest.json'}
expected |= {f'data/{kind}_{split}.jsonl' for kind in ('ner','linkage') for split in ('train','validation','test')}
with ZipFile(io.BytesIO(uploaded[name])) as z:
    for rel in sorted(expected):
        src = 'training/' + rel
        assert src in z.namelist(), f'Missing {src}'
        p = work / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        with z.open(src) as a, p.open('wb') as b: shutil.copyfileobj(a,b)
del uploaded
os.chdir(work)
print('Training source and datasets extracted.')

In [ ]:
# 3. Check dataset integrity and print only aggregate counts.
import json
from train_models import verify
manifest = verify(work / 'data')
for split, kinds in manifest['splits'].items():
    print(split, {kind: info['count'] for kind, info in kinds.items()})

In [ ]:
# 4. Train both models. No test data is used for choosing the linkage threshold.
# The CRF uses fixed hyperparameters. CPU is sufficient; GPU not needed.
export_root = work / 'export'
model_dir = export_root / 'models'
if export_root.exists(): shutil.rmtree(export_root)
subprocess.check_call([sys.executable, 'train_models.py', '--data', 'data', '--output', str(model_dir), '--iterations', '65'])

In [ ]:
# 5. Display measured evaluation results, not invented accuracy.
import pandas as pd
metrics = json.loads((model_dir / 'evaluation_metrics.json').read_text())
summary=[]
for split in ('validation','test'):
    ner=metrics['ner'][split]
    link=metrics['linkage'][split]
    summary += [dict(model='CRF exact PII spans',split=split,**{k:ner['pii_exact_span_micro'][k] for k in ('precision','recall','f1')}), dict(model='Linkage candidate pairs',split=split,**{k:link[k] for k in ('precision','recall','f1')})]
display(pd.DataFrame(summary))
display(pd.DataFrame(metrics['ner']['test']['per_language']).T)
display(pd.DataFrame(metrics['ner']['test']['per_label']).T)
print('Protected span overlap rate (test):', metrics['ner']['test']['protected_overlap_rate'])
print('Selected linkage threshold:', metrics['linkage']['threshold_selected_on_validation'])
print('These are synthetic held-out metrics, not official evaluator scores.')

In [ ]:
# 6. Reload artifacts and test the runtime interface on a synthetic example.
from model_runtime import CandidateModels
runtime = CandidateModels(model_dir)
assert isinstance(runtime.spans('Patient name: Example Person; BP 120/80 mmHg.'), list)
assert runtime.pair({'strong_id':'SYN-A'}, {'strong_id':'SYN-B'})['candidate_link'] is False
assert runtime.pair({'name':'Example Person','phone':'0000000000'}, {'name':'Example Person','phone':'0000000000'})['candidate_link'] is False
print('Artifact reload and conservative pair checks passed.')

In [ ]:
# 7. Export only model artifacts and their inference adapters.
from zipfile import ZIP_DEFLATED
required = ['pii_crf.crfsuite','linkage_parameters.json','evaluation_metrics.json','model_metadata.json','model_features.py','model_runtime.py']
assert all((model_dir / p).is_file() and (model_dir / p).stat().st_size > 0 for p in required)
out = Path('/content/Sheild_Trained_Models.zip')
with ZipFile(out,'w',compression=ZIP_DEFLATED) as z:
    for p in required: z.write(model_dir/p, 'models/'+p)
print('Exported:', out.name)
files.download(str(out))

## Next step
Extract `Sheild_Trained_Models.zip` into the existing Sheild root. Append `python-crfsuite==0.9.12` to the existing root requirements.txt; do not replace the file. Retain training source and model metadata. Integrate model spans as candidates in Phase 2C and pair scores in Phase 3; never bypass the independent release verifier. Do not claim official scores based on this notebook.